**2026-10-09 update:** This collection contains repeated `vid_<hash>_face_...` filename families. The original per-image manifests permit family overlap. Follow `RGB_GENERALIZATION_REPAIR.md` before another training run; keep existing hashes/backups. The repaired split does not by itself remove source/format bias.

# Colab session 2 — single-GPU RGB training

Start a NEW Colab session with one CUDA GPU (for example T4, L4, or A100). If “G4” was a typo for T4, T4 works too; the setup detects the actual device and VRAM. GPU availability varies ([Colab FAQ](https://research.google.com/colaboratory/faq.html)). No distributed/two-GPU launch is needed.

Restore the SAME image dataset at `/content/dataset/Preapred Dataset/train` each session. Restoring manifests alone is insufficient. These cells reuse completed SHA256 evidence under the explicit unchanged-dataset assumption; they do not claim to reverify current bytes. Dataset changes require renewed verification.

Select `TRAIN_SIZE='100000'` first; later use `500000`, `1000000`, or `all`. Read actual counts before training. Use the same shared development cohort and a fresh run directory for each experiment. No old model checkpoint is required. CPU preparation never needs to run again for these saved cohorts while image bytes and paths remain unchanged.

The RGB R2 recipe is a starting experiment, not an accuracy guarantee. Full GPU utilization cannot be guaranteed: benchmark on the assigned CPU/GPU and local dataset first. Code/runtime tests below were not executed locally.


## 1. Mount Drive and set paths

Same paths as the CPU notebook.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, shutil, subprocess, sys, hashlib, zipfile
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/deepfake_rgb')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
REPO_ROOT = Path('/content/Novel-Deepfake-Detection')
DATA_ROOT = Path('/content/dataset/Preapred Dataset/train')
PREP_ROOT = Path('/content/prepared_data/rgb_sizes_seed42')
os.environ.update(REPO_ROOT=str(REPO_ROOT), DATA_ROOT=str(DATA_ROOT), PREP_ROOT=str(PREP_ROOT))
print('Persistent storage:', DRIVE_ROOT)
print('Required images:', DATA_ROOT)


## 2. Restore prepared manifests and exact code

Do not clone another branch over this code snapshot. Use a fresh /content session.


In [ ]:
PREP_ROOT.mkdir(parents=True, exist_ok=True)
for archive_name, destination in [('prepared_rgb_sizes_seed42.zip', PREP_ROOT),
                                  ('prepared_rgb_code.zip', REPO_ROOT)]:
    archive_path = DRIVE_ROOT / archive_name
    assert archive_path.is_file(), f'Complete CPU preparation first: {archive_path}'
    assert not destination.exists() or not any(destination.iterdir()), f'Use a fresh session/empty destination: {destination}'
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(destination)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('Restored preparation and the exact saved code. No inventory or hash run required.')


## 3. Dependencies

Keep Colab’s preinstalled compatible torch/torchvision pair.


In [ ]:
%pip install -q numpy pillow pytorch-wavelets PyWavelets tensorboard pytest


## 4. Restore original images

Repeat your original dataset download/extraction now. No hash/inventory command is needed for unchanged images.


In [ ]:
# Put/download/extract your original dataset here before running this cell.
# Required: /content/dataset/Preapred Dataset/train/real and .../fake
# Also supported by inventory: 0_real and 1_fake instead of real and fake.
assert DATA_ROOT.is_dir(), f'Extract the image dataset first: {DATA_ROOT}'
class_dirs = [p.name for p in DATA_ROOT.iterdir() if p.is_dir()]
assert ({'real', 'fake'} <= set(class_dirs) or {'0_real', '1_fake'} <= set(class_dirs)), class_dirs
print('Class folders:', class_dirs)
print('Free local disk (GiB):', round(shutil.disk_usage('/content').free / 2**30, 1))
print('CPU count:', os.cpu_count())
print('RAM:', next(line.strip() for line in Path('/proc/meminfo').read_text().splitlines() if line.startswith('MemTotal:')))


## 5. Choose size and detect one GPU

Batch/accumulation keep effective batch 64. For an out-of-memory error, halve BATCH and double ACCUM before benchmarking again. Change os.environ values too if adjusting after this cell.


In [ ]:
import torch
from uuid import uuid4
from data.manifest import verify_manifest_gate

TRAIN_SIZE = '100000'  # '500000', '1000000', or 'all' in later sessions
UNCHANGED_DATASET = True  # True only for the SAME image bytes and paths as session 1
assert UNCHANGED_DATASET, 'Changed data needs renewed preparation/audit.'
assert torch.cuda.is_available(), 'Select a GPU runtime first.'
manifest = PREP_ROOT / 'sizes' / TRAIN_SIZE / 'selected_manifest.csv'
verify_manifest_gate(manifest, require_hashes=True)
summary = json.loads((manifest.parent / 'selection_report.json').read_text())
print('Training size actually available:', summary['actual_training_samples'])
print('Shared dev size:', summary['dev_samples'], 'Training shortfall:', summary['shortfall'])
props = torch.cuda.get_device_properties(0)
vram_gib = props.total_memory / 2**30
# Starting points, not throughput guarantees; use the benchmark below.
batch = 64 if vram_gib >= 35 else (32 if vram_gib >= 20 else 16)
accum = 64 // batch  # same effective batch of 64 on one GPU
workers = min(4, max(1, (os.cpu_count() or 2) - 1))
run_id = 'seed42_' + uuid4().hex[:8]
output = Path('/content/rgb_pilots')
output.mkdir(exist_ok=True)
values = dict(REPO_ROOT=str(REPO_ROOT), DATA_ROOT=str(DATA_ROOT), MANIFEST=str(manifest),
    VAL_MANIFEST=str(manifest), TRAIN_SPLIT='train', DEV_SPLIT='dev', GPU_COUNT='1', GPU_IDS='0',
    BATCH=str(batch), ACCUM=str(accum), WORKERS=str(workers), OUTPUT_BASE=str(output),
    RUN_ID=run_id, RUN_NAME='rgb_r2_' + TRAIN_SIZE, RGB_HEAD='128d', RGB_DROPOUT='0.5',
    FINE_TUNE='layer4_and_head', BN_POLICY='frozen', LR_MULT='0.1', AUG_RECIPE='rgb_v1',
    SCIENCE_MODE='engineering', ELIGIBLE_SOURCES='')
run_dir = output / (values['RUN_NAME'] + '_' + run_id)
values.update(RGB_RUN_DIR=str(run_dir), RGB_CKPT=str(run_dir / 'checkpoints/best.pth'))
os.environ.pop('RESUME_CHECKPOINT', None)
os.environ.update(values)
(output / (run_id + '_session.json')).write_text(json.dumps(values, indent=2))
print('GPU:', props.name, '| VRAM GiB:', round(vram_gib, 1))
print('Batch:', batch, '| accumulation:', accum, '| workers:', workers)
print('Run:', run_dir)
print('Do not rerun this setup cell during the same experiment; it creates a new run ID.')


## 6. Focused code checks

Stop if checks fail; synthetic parity uses a tiny synthetic input, not your external test.


In [ ]:
%%bash
set -euo pipefail
cd "$REPO_ROOT"
python -m pytest -q tests/test_colab_manifest_sizes.py tests/test_prepared_manifest_recovery.py tests/test_rgb_review_final_repairs.py \
  tests/test_versioned_augmentations.py tests/test_calibration_and_source_readiness.py \
  tests/test_round1_phase3_repairs.py
python -u tools/diagnose_rgb_parity.py --synthetic --arch Wang2020_128 \
  --output_report "$OUTPUT_BASE/${RUN_ID}_synthetic_parity.json"


## 7. Benchmark

Inspect throughput, data-wait and GPU timing before full training. If CPU loading dominates, try WORKERS=2 versus 4 within available CPUs; rerun this bounded benchmark. Keep inputs on local disk. Update os.environ["BATCH"], ["ACCUM"], ["WORKERS"] to change settings. No performance promise from a GPU name alone.


In [ ]:
%%bash
set -euo pipefail
cd "$REPO_ROOT"
PYTHONUNBUFFERED=1 python -u tools/benchmark_training_throughput.py \
  --manifest "$MANIFEST" --val_manifest "$VAL_MANIFEST" --dataroot "$DATA_ROOT" \
  --gpu_ids "$GPU_IDS" --batch_size "$BATCH" --grad_accum_steps "$ACCUM" \
  --num_workers "$WORKERS" --prefetch_factor 2 --persistent_workers --pin_memory \
  --use_amp --amp_dtype fp16 --aug_recipe "$AUG_RECIPE" \
  --fine_tune_policy "$FINE_TUNE" --bn_policy "$BN_POLICY" \
  --rgb_head_type "$RGB_HEAD" --rgb_dropout "$RGB_DROPOUT" --backbone_lr_mult "$LR_MULT" \
  --warmup_microbatches 20 --measure_microbatches 100 --val_samples 500 \
  --output "$OUTPUT_BASE/${RUN_ID}_benchmark.json"


## 8. Start periodic Drive backups

Best effort every five minutes; a sudden runtime loss can lose work since the last backup. Final ZIP below is the complete saved result. Stop training before restoring a backup; do not rerun setup to resume.


In [ ]:
# Periodically sync stable completed files; run before training.
# Local images/checkpoints stay fast; copies go to Drive in the background.
import threading, time
if 'training_backup_stop' in globals():
    training_backup_stop.set()
    training_backup_thread.join()
training_backup_stop = threading.Event()
remote_run = DRIVE_ROOT / 'runs' / (values['RUN_NAME'] + '_' + values['RUN_ID'])
remote_run.mkdir(parents=True, exist_ok=True)

def sync_results():
    local_run = Path(os.environ['RGB_RUN_DIR'])
    if not local_run.exists():
        return
    for source in local_run.rglob('*'):
        if not source.is_file() or source.name.endswith(('.tmp', '.temp', '.uploading')):
            continue
        before = source.stat()
        if time.time() - before.st_mtime < 15:
            continue
        target = remote_run / source.relative_to(local_run)
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists() and target.stat().st_size == before.st_size and abs(target.stat().st_mtime - before.st_mtime) < 1:
            continue
        tmp = target.with_name(target.name + '.uploading')
        shutil.copy2(source, tmp)
        after = source.stat()
        if (before.st_size, before.st_mtime_ns) == (after.st_size, after.st_mtime_ns):
            tmp.replace(target)
    shutil.copy2(Path(os.environ['OUTPUT_BASE']) / (values['RUN_ID'] + '_session.json'), remote_run / 'session.json')

def training_backup_worker():
    while not training_backup_stop.wait(300):
        try:
            sync_results()
            print('Stable training outputs copied to Drive.', flush=True)
        except Exception as exc:
            print('Training backup failed; local training continues:', exc, flush=True)
training_backup_thread = threading.Thread(target=training_backup_worker, daemon=True)
training_backup_thread.start()
print('Backup destination:', remote_run)


## 9. Train RGB

One GPU, up to five epochs, early stopping, pretrained RGB backbone. Development images never enter training. Output logs remain visible.


In [ ]:
%%bash
set -euo pipefail
cd "$REPO_ROOT"
source_args=()
if [[ "$SCIENCE_MODE" == "engineering" ]]; then
  source_args=(--allow_aggregate_sources --allow_source_overlap --monitor_metric auc)
elif [[ "$SCIENCE_MODE" == "heldout" ]]; then
  : "${ELIGIBLE_SOURCES:?Declare verified held-out development sources}"
  source_args=(--monitor_metric source_macro_auc --eligible_sources "$ELIGIBLE_SOURCES")
else
  echo "Unknown SCIENCE_MODE" >&2; exit 1
fi
aug_args=()
if [[ "$AUG_RECIPE" == "legacy" ]]; then
  aug_args=(--blur_prob 0.5 --blur_sig 0.0,3.0 --jpg_prob 0)
fi
resume_args=()
if [[ -n "${RESUME_CHECKPOINT:-}" ]]; then
  resume_args=(--continue_train --resume_checkpoint "$RESUME_CHECKPOINT")
fi
PYTHONUNBUFFERED=1 python -u train.py \
  --arch Wang2020_128 --name "$RUN_NAME" --run_id "$RUN_ID" --checkpoints_dir "$OUTPUT_BASE" \
  --dataroot "$DATA_ROOT" --manifest "$MANIFEST" --manifest_split "$TRAIN_SPLIT" \
  --val_manifest "$VAL_MANIFEST" --val_manifest_split "$DEV_SPLIT" \
  --rgb_head_type "$RGB_HEAD" --rgb_dropout "$RGB_DROPOUT" \
  --fine_tune_policy "$FINE_TUNE" --bn_policy "$BN_POLICY" --backbone_lr_mult "$LR_MULT" \
  --aug_recipe "$AUG_RECIPE" "${aug_args[@]}" \
  --gpu_ids "$GPU_IDS" --batch_size "$BATCH" --grad_accum_steps "$ACCUM" \
  --val_batch_size "$BATCH" --num_workers "$WORKERS" --prefetch_factor 2 --persistent_workers --pin_memory \
  --epochs 5 --epochs_decay 0 --optim adam --lr 0.0001 --lr_policy cosine \
  --use_amp --amp_dtype fp16 --val_precision fp32 --seed 42 --pretrained \
  --early_stopping --early_stopping_patience 2 --early_stopping_min_epochs 2 \
  --save_epoch_freq 0 "${source_args[@]}" "${resume_args[@]}"


## 10. Calibrate on development predictions

Never choose a threshold using final external-test labels. The same selection dev is used here; this is not independent calibration evidence.


In [ ]:
%%bash
set -euo pipefail
cd "$REPO_ROOT"
python analyze_predictions.py calibrate \
  --predictions "$RGB_RUN_DIR/checkpoints/best_dev_predictions.csv" \
  --output "$RGB_RUN_DIR/checkpoints/threshold.json"


## 11. Save complete results to Drive

Wait for the final saved confirmation before disconnecting. Then evaluate the RGB best checkpoint on unseen data using its saved threshold, without tuning on the final test.


In [ ]:
training_backup_stop.set()
training_backup_thread.join()
local_run = Path(os.environ['RGB_RUN_DIR'])
assert (local_run / 'checkpoints/best.pth').is_file()
assert (local_run / 'checkpoints/threshold.json').is_file()
archive = shutil.make_archive('/content/' + local_run.name, 'zip', root_dir=local_run)
destination = DRIVE_ROOT / 'runs' / (local_run.name + '.zip')
temporary = destination.with_suffix('.uploading')
shutil.copy2(archive, temporary)
temporary.replace(destination)
shutil.copy2(Path(os.environ['OUTPUT_BASE']) / (values['RUN_ID'] + '_session.json'), remote_run / 'session.json')
for report in Path(os.environ['OUTPUT_BASE']).glob(values['RUN_ID'] + '_*.json'):
    shutil.copy2(report, remote_run / report.name)
print('FINAL MODEL, THRESHOLD, LOGS AND PREDICTIONS SAVED:', destination)
print('Keep the prepared data ZIP and original image dataset for future training sizes.')
